# Исторический исследовательский код

Эта версия не является основным протоколом статьи. Выводы удалены. Она может требовать исходные локальные файлы и иную схему данных, которые не включены в открытый набор. Для воспроизведения статьи используйте `notebooks/vascular`. Старые метрики не смешиваются с актуальными.


# Сравнение способов балансировки госпитальной смертности

В ноутбуке сравниваются три варианта при одинаковом отношении умерших к выжившим `0,20`:

1. RandomOverSampler как контроль дублирования.
2. SMOTENC с двумя ближайшими соседями.
3. SMOTENC с тремя ближайшими соседями.

Каждый вариант повторяется 100 раз. Метод выбирается по сохранению распределений, долей категориальных факторов, корреляций, пропусков и величин эффектов. Прогностическая модель не строится. Итоговые таблицы и графики выводятся в ноутбуке; сохраняется только один финальный сбалансированный датасет.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown
from scipy.stats import mannwhitneyu, fisher_exact, wasserstein_distance
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import RandomOverSampler, SMOTENC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")

DATA_PATH = Path("data/hospital_outcomes_prepared.csv")
FINAL_PATH = Path("data/hospital_outcomes_balanced_final.csv")
SAMPLING_STRATEGY = 0.20
N_RUNS = 100
BASE_SEED = 2026

## 1. Исходная выборка

Балансировка выполняется на копии подготовленного датасета. Реальные строки не изменяются и сохраняют исходные пропуски.

In [ ]:
real_data = pd.read_csv(DATA_PATH)

cohort_overview = pd.DataFrame(
    {
        "Показатель": [
            "Всего пациентов",
            "Выживших",
            "Умерших",
            "Госпитальная смертность, %",
        ],
        "Значение": [
            len(real_data),
            int((real_data["hospital_death"] == 0).sum()),
            int((real_data["hospital_death"] == 1).sum()),
            real_data["hospital_death"].mean() * 100,
        ],
    }
)

display(cohort_overview)
display(
    real_data
    .groupby("vascular_group", observed=True)["hospital_death"]
    .agg(["size", "sum", "mean"])
    .rename(columns={"size": "patients", "sum": "deaths", "mean": "mortality"})
)

## 2. Признаки для генерации

В генерацию включаются исходные числовые и категориальные признаки. Категория ФВ, ХБП и повышенная глюкоза рассчитываются после генерации, чтобы не создавать противоречивые сочетания.

In [ ]:
continuous_generation_features = [
    "age_years",
    "ef_percent",
    "creatinine_value",
    "total_cholesterol",
    "ldl_mmol_l",
    "glucose_value",
    "crp_value",
    "egfr_value",
    "bmi",
]

categorical_generation_features = [
    "vascular_group",
    "female",
    "prior_vascular_surgery",
    "mi_history",
    "stroke_history",
]

continuous_analysis_features = continuous_generation_features

binary_analysis_features = [
    "female",
    "prior_vascular_surgery",
    "mi_history",
    "stroke_history",
    "ckd_g3plus_proxy",
    "glucose_elevated_proxy",
]

categorical_analysis_features = [
    "vascular_group",
    *binary_analysis_features,
    "ef_category",
]

feature_labels = {
    "age_years": "Возраст",
    "ef_percent": "Фракция выброса",
    "creatinine_value": "Креатинин",
    "total_cholesterol": "Общий холестерин",
    "ldl_mmol_l": "ЛПНП",
    "glucose_value": "Глюкоза",
    "crp_value": "С-реактивный белок",
    "egfr_value": "СКФ",
    "bmi": "Индекс массы тела",
    "vascular_group": "Сосудистый бассейн",
    "female": "Женский пол",
    "prior_vascular_surgery": "Операция на сосудах в анамнезе",
    "mi_history": "Инфаркт миокарда в анамнезе",
    "stroke_history": "ОНМК в анамнезе",
    "ckd_g3plus_proxy": "ХБП G3 и выше",
    "glucose_elevated_proxy": "Глюкоза ≥5,6 ммоль/л",
    "ef_category": "Категория ФВ",
}

## 3. Подготовка матрицы и генерация

Пропуски временно заполняются медианами по сосудистому бассейну и полу. Для них создаются отдельные индикаторы. После генерации пропуски восстанавливаются. Числовые признаки масштабируются устойчивым к выбросам способом.

In [ ]:
def prepare_generation_matrix(frame):
    matrix = frame[
        continuous_generation_features
        + categorical_generation_features
    ].copy()

    missing_indicator_columns = []
    indicator_to_variable = {}

    for variable in continuous_generation_features:
        if frame[variable].isna().any():
            indicator = f"{variable}_was_missing"
            missing_indicator_columns.append(indicator)
            indicator_to_variable[indicator] = variable
            matrix[indicator] = frame[variable].isna().astype(int)

            group_sex_median = (
                frame
                .groupby(["vascular_group", "female"], observed=True)[variable]
                .transform("median")
            )
            group_median = (
                frame
                .groupby("vascular_group", observed=True)[variable]
                .transform("median")
            )
            matrix[variable] = (
                frame[variable]
                .fillna(group_sex_median)
                .fillna(group_median)
                .fillna(frame[variable].median())
            )

    scaler = RobustScaler()
    matrix[continuous_generation_features] = scaler.fit_transform(
        matrix[continuous_generation_features]
    )

    categorical_columns = (
        categorical_generation_features
        + missing_indicator_columns
    )

    for variable in categorical_columns:
        matrix[variable] = matrix[variable].astype("category")

    return {
        "matrix": matrix,
        "scaler": scaler,
        "categorical_columns": categorical_columns,
        "missing_indicator_columns": missing_indicator_columns,
        "indicator_to_variable": indicator_to_variable,
    }


generation_setup = prepare_generation_matrix(real_data)
generation_matrix = generation_setup["matrix"]
target = real_data["hospital_death"].astype(int)

print("Матрица генерации:", generation_matrix.shape)
print("Категориальных столбцов:", len(generation_setup["categorical_columns"]))
print("Индикаторов пропусков:", len(generation_setup["missing_indicator_columns"]))

In [ ]:
def derive_features(frame):
    result = frame.copy()

    result["ef_category"] = pd.Series(
        pd.NA,
        index=result.index,
        dtype="string"
    )
    result.loc[result["ef_percent"] < 40, "ef_category"] = "<40"
    result.loc[
        result["ef_percent"].between(40, 50, inclusive="both"),
        "ef_category"
    ] = "40–50"
    result.loc[result["ef_percent"] > 50, "ef_category"] = ">50"

    result["ckd_g3plus_proxy"] = pd.Series(
        pd.NA,
        index=result.index,
        dtype="Int64"
    )
    has_egfr = result["egfr_value"].notna()
    result.loc[has_egfr, "ckd_g3plus_proxy"] = (
        result.loc[has_egfr, "egfr_value"] < 60
    ).astype(int)

    result["glucose_elevated_proxy"] = pd.Series(
        pd.NA,
        index=result.index,
        dtype="Int64"
    )
    has_glucose = result["glucose_value"].notna()
    result.loc[has_glucose, "glucose_elevated_proxy"] = (
        result.loc[has_glucose, "glucose_value"] >= 5.6
    ).astype(int)

    return result


def sampler_for(method, seed, sampling_strategy=SAMPLING_STRATEGY):
    if method == "RandomOverSampler":
        return RandomOverSampler(
            sampling_strategy=sampling_strategy,
            random_state=seed,
        )
    if method == "SMOTENC_k2":
        return SMOTENC(
            categorical_features=generation_setup["categorical_columns"],
            sampling_strategy=sampling_strategy,
            k_neighbors=2,
            random_state=seed,
        )
    if method == "SMOTENC_k3":
        return SMOTENC(
            categorical_features=generation_setup["categorical_columns"],
            sampling_strategy=sampling_strategy,
            k_neighbors=3,
            random_state=seed,
        )
    raise ValueError(method)


def generate_balanced_dataset(
    method,
    seed,
    sampling_strategy=SAMPLING_STRATEGY,
):
    sampler = sampler_for(method, seed, sampling_strategy)
    resampled_matrix, resampled_target = sampler.fit_resample(
        generation_matrix,
        target,
    )

    synthetic = resampled_matrix.iloc[len(real_data):].copy()
    synthetic[continuous_generation_features] = (
        generation_setup["scaler"]
        .inverse_transform(synthetic[continuous_generation_features])
    )

    for indicator in generation_setup["missing_indicator_columns"]:
        variable = generation_setup["indicator_to_variable"][indicator]
        restore_missing = (
            pd.to_numeric(synthetic[indicator], errors="coerce")
            .round()
            .eq(1)
        )
        synthetic.loc[restore_missing, variable] = np.nan

    synthetic = synthetic[
        continuous_generation_features
        + categorical_generation_features
    ].copy()

    synthetic["age_years"] = synthetic["age_years"].round()

    for variable in [
        "female",
        "prior_vascular_surgery",
        "mi_history",
        "stroke_history",
    ]:
        synthetic[variable] = (
            pd.to_numeric(synthetic[variable], errors="coerce")
            .round()
            .astype("Int64")
        )

    synthetic["vascular_group"] = synthetic["vascular_group"].astype("string")
    synthetic = derive_features(synthetic)
    synthetic["hospital_death"] = 1
    synthetic["is_synthetic"] = 1
    synthetic["synthetic_method"] = method
    synthetic["synthetic_seed"] = seed
    synthetic["synthetic_sampling_strategy"] = sampling_strategy

    real_part = real_data.copy()
    real_part["is_synthetic"] = 0
    real_part["synthetic_method"] = "real"
    real_part["synthetic_seed"] = pd.NA
    real_part["synthetic_sampling_strategy"] = pd.NA

    final_columns = [
        *real_data.columns,
        "is_synthetic",
        "synthetic_method",
        "synthetic_seed",
        "synthetic_sampling_strategy",
    ]

    balanced = pd.concat(
        [real_part[final_columns], synthetic[final_columns]],
        ignore_index=True,
    )

    return balanced, synthetic.reset_index(drop=True)

## 4. Проверка одной генерации

До Монте-Карло проверяются размер наборов, число добавленных умерших и корректность технических полей.

In [ ]:
generation_check_rows = []

for method in ["RandomOverSampler", "SMOTENC_k2", "SMOTENC_k3"]:
    balanced_check, synthetic_check = generate_balanced_dataset(
        method,
        BASE_SEED,
    )
    generation_check_rows.append(
        {
            "method": method,
            "total_rows": len(balanced_check),
            "real_deaths": int(
                (
                    (balanced_check["hospital_death"] == 1)
                    & (balanced_check["is_synthetic"] == 0)
                ).sum()
            ),
            "synthetic_deaths": len(synthetic_check),
            "all_deaths": int(balanced_check["hospital_death"].sum()),
            "death_to_alive_ratio": (
                balanced_check["hospital_death"].sum()
                / (balanced_check["hospital_death"] == 0).sum()
            ),
        }
    )

generation_check = pd.DataFrame(generation_check_rows)
display(generation_check)

## 5. Метрики статистической близости

Качество оценивается по величинам различий, а не по уменьшению p-value. Дополнительно контролируются пропуски, корреляции, дубликаты и простые допустимые диапазоны.

In [ ]:
def pooled_smd(real_values, synthetic_values):
    real_values = pd.Series(real_values).dropna().astype(float)
    synthetic_values = pd.Series(synthetic_values).dropna().astype(float)

    if len(real_values) < 2 or len(synthetic_values) < 2:
        return np.nan

    pooled_variance = (
        (real_values.var(ddof=1) + synthetic_values.var(ddof=1))
        / 2
    )
    if pooled_variance <= 0:
        return 0.0
    return (
        synthetic_values.mean() - real_values.mean()
    ) / np.sqrt(pooled_variance)


def normalized_wasserstein(real_values, synthetic_values):
    real_values = pd.Series(real_values).dropna().astype(float)
    synthetic_values = pd.Series(synthetic_values).dropna().astype(float)

    if real_values.empty or synthetic_values.empty:
        return np.nan

    scale = real_values.quantile(0.75) - real_values.quantile(0.25)
    if not np.isfinite(scale) or scale == 0:
        scale = real_values.std(ddof=1)
    if not np.isfinite(scale) or scale == 0:
        scale = 1.0

    return wasserstein_distance(real_values, synthetic_values) / scale


def categorical_distance(real_series, synthetic_series):
    real_distribution = (
        real_series.astype("string").fillna("<NA>").value_counts(normalize=True)
    )
    synthetic_distribution = (
        synthetic_series.astype("string").fillna("<NA>").value_counts(normalize=True)
    )
    categories = real_distribution.index.union(synthetic_distribution.index)
    return 0.5 * sum(
        abs(
            real_distribution.get(category, 0)
            - synthetic_distribution.get(category, 0)
        )
        for category in categories
    )


def correlation_distance(real_frame, synthetic_frame):
    real_correlation = real_frame[continuous_analysis_features].corr(
        method="spearman",
        min_periods=5,
    )
    synthetic_correlation = synthetic_frame[continuous_analysis_features].corr(
        method="spearman",
        min_periods=5,
    )
    difference = (real_correlation - synthetic_correlation).abs()
    upper_triangle = np.triu(
        np.ones(difference.shape, dtype=bool),
        k=1,
    )
    values = difference.where(upper_triangle).stack()
    return values.mean() if not values.empty else np.nan


def rank_biserial_effect(frame, variable):
    alive = frame.loc[frame["hospital_death"] == 0, variable].dropna()
    deceased = frame.loc[frame["hospital_death"] == 1, variable].dropna()
    if len(alive) == 0 or len(deceased) == 0:
        return np.nan
    statistic = mannwhitneyu(
        deceased,
        alive,
        alternative="two-sided",
    ).statistic
    return 2 * statistic / (len(deceased) * len(alive)) - 1


def log_odds_effect(frame, variable):
    observed = frame[[variable, "hospital_death"]].dropna()
    a = int(((observed[variable] == 1) & (observed["hospital_death"] == 1)).sum())
    b = int(((observed[variable] == 1) & (observed["hospital_death"] == 0)).sum())
    c = int(((observed[variable] == 0) & (observed["hospital_death"] == 1)).sum())
    d = int(((observed[variable] == 0) & (observed["hospital_death"] == 0)).sum())
    return np.log(((a + 0.5) * (d + 0.5)) / ((b + 0.5) * (c + 0.5)))


real_deaths = real_data.loc[real_data["hospital_death"] == 1].copy()

reference_effects = {
    **{
        variable: rank_biserial_effect(real_data, variable)
        for variable in continuous_analysis_features
    },
    **{
        variable: log_odds_effect(real_data, variable)
        for variable in binary_analysis_features
    },
}

plausible_ranges = {
    "age_years": (18, 105),
    "ef_percent": (10, 90),
    "creatinine_value": (0, 1500),
    "total_cholesterol": (0, 15),
    "ldl_mmol_l": (0, 15),
    "glucose_value": (2, 40),
    "crp_value": (0, 500),
    "egfr_value": (1, 200),
    "bmi": (10, 80),
}


def standardized_row_hash(frame):
    comparable = frame[
        continuous_generation_features
        + categorical_generation_features
    ].copy()
    for variable in continuous_generation_features:
        comparable[variable] = (
            pd.to_numeric(comparable[variable], errors="coerce")
            .round(6)
            .astype("Float64")
            .astype("string")
        )
    comparable["vascular_group"] = (
        comparable["vascular_group"]
        .astype("string")
        .str.strip()
    )
    for variable in [
        "female",
        "prior_vascular_surgery",
        "mi_history",
        "stroke_history",
    ]:
        comparable[variable] = (
            pd.to_numeric(comparable[variable], errors="coerce")
            .round()
            .astype("Int64")
            .astype("string")
        )
    comparable = comparable.fillna("<NA>")
    return pd.util.hash_pandas_object(comparable, index=False)


def evaluate_generation(
    method,
    seed,
    sampling_strategy=SAMPLING_STRATEGY,
):
    balanced, synthetic = generate_balanced_dataset(
        method,
        seed,
        sampling_strategy,
    )

    smd_values = [
        abs(pooled_smd(real_deaths[variable], synthetic[variable]))
        for variable in continuous_analysis_features
    ]
    wasserstein_values = [
        normalized_wasserstein(
            real_deaths[variable],
            synthetic[variable],
        )
        for variable in continuous_analysis_features
    ]
    categorical_values = [
        categorical_distance(
            real_deaths[variable],
            synthetic[variable],
        )
        for variable in categorical_analysis_features
    ]
    missingness_values = [
        abs(
            real_deaths[variable].isna().mean()
            - synthetic[variable].isna().mean()
        )
        for variable in continuous_analysis_features
    ]

    effect_values = []
    for variable in continuous_analysis_features:
        current_effect = rank_biserial_effect(balanced, variable)
        effect_values.append(abs(current_effect - reference_effects[variable]))
    for variable in binary_analysis_features:
        current_effect = log_odds_effect(balanced, variable)
        effect_values.append(abs(current_effect - reference_effects[variable]))

    violations = 0
    for variable, (lower, upper) in plausible_ranges.items():
        violations += int(
            (
                synthetic[variable].notna()
                & ~synthetic[variable].between(lower, upper)
            ).sum()
        )

    invalid_binary = 0
    for variable in [
        "female",
        "prior_vascular_surgery",
        "mi_history",
        "stroke_history",
        "ckd_g3plus_proxy",
        "glucose_elevated_proxy",
    ]:
        invalid_binary += int(
            (
                synthetic[variable].notna()
                & ~synthetic[variable].isin([0, 1])
            ).sum()
        )

    synthetic_hashes = standardized_row_hash(synthetic)
    real_hashes = standardized_row_hash(real_deaths)
    duplicate_rate = synthetic_hashes.isin(set(real_hashes)).mean()

    carotid_share = (
        synthetic["vascular_group"].eq("carotid").mean()
    )

    return {
        "method": method,
        "seed": seed,
        "sampling_strategy": sampling_strategy,
        "smd_mean": np.nanmean(smd_values),
        "wasserstein_mean": np.nanmean(wasserstein_values),
        "categorical_difference": np.nanmean(categorical_values),
        "missingness_difference": np.nanmean(missingness_values),
        "correlation_difference": correlation_distance(real_deaths, synthetic),
        "effect_difference": np.nanmean(effect_values),
        "duplicate_rate": duplicate_rate,
        "range_violations": violations,
        "invalid_binary": invalid_binary,
        "synthetic_carotid_share": carotid_share,
    }

## 6. Сто повторных генераций для каждого метода

In [ ]:
methods = ["RandomOverSampler", "SMOTENC_k2", "SMOTENC_k3"]
simulation_rows = []

for method in methods:
    for run in range(N_RUNS):
        simulation_rows.append(
            evaluate_generation(
                method=method,
                seed=BASE_SEED + run,
            )
        )

simulation_results = pd.DataFrame(simulation_rows)

print("Выполнено генераций:", len(simulation_results))
display(simulation_results.head())

## 7. Сводное сравнение методов

Меньшие значения метрик означают лучшее сохранение структуры реальных умерших. `duplicate_rate` ожидаемо высок для RandomOverSampler и используется только как контроль разнообразия.

In [ ]:
summary_metrics = [
    "smd_mean",
    "wasserstein_mean",
    "categorical_difference",
    "missingness_difference",
    "correlation_difference",
    "effect_difference",
    "duplicate_rate",
    "range_violations",
    "invalid_binary",
    "synthetic_carotid_share",
]

method_summary = (
    simulation_results
    .groupby("method", observed=True)[summary_metrics]
    .agg(["median", "mean", "std"])
)

display(method_summary)

In [ ]:
plot_metrics = [
    "smd_mean",
    "wasserstein_mean",
    "categorical_difference",
    "correlation_difference",
    "effect_difference",
    "duplicate_rate",
]

figure, axes = plt.subplots(2, 3, figsize=(18, 10))

for axis, metric in zip(axes.flat, plot_metrics):
    sns.boxplot(
        data=simulation_results,
        x="method",
        y=metric,
        hue="method",
        palette="Set2",
        legend=False,
        ax=axis,
    )
    axis.set_title(metric)
    axis.set_xlabel("")
    axis.tick_params(axis="x", rotation=15)

figure.suptitle("Устойчивость статистических характеристик в 100 генерациях", y=1.02)
figure.tight_layout()
plt.show()
plt.close(figure)

## 8. Выбор конфигурации SMOTENC

RandomOverSampler остаётся контрольным вариантом, поскольку не создаёт новых клинических профилей. Между двумя вариантами SMOTENC выбирается конфигурация с меньшим средним рангом по статистическим метрикам. Затем выбирается типичная, а не наиболее удачная случайная реализация.

In [ ]:
selection_metrics = [
    "smd_mean",
    "wasserstein_mean",
    "categorical_difference",
    "missingness_difference",
    "correlation_difference",
    "effect_difference",
]

smotenc_results = simulation_results.loc[
    simulation_results["method"].isin(["SMOTENC_k2", "SMOTENC_k3"])
].copy()

eligible_results = smotenc_results.loc[
    (smotenc_results["range_violations"] == 0)
    & (smotenc_results["invalid_binary"] == 0)
].copy()

if eligible_results.empty:
    raise RuntimeError("Все реализации SMOTENC нарушили технические ограничения")

selection_summary = (
    eligible_results
    .groupby("method", observed=True)[selection_metrics]
    .median()
)

for metric in selection_metrics:
    selection_summary[f"{metric}_rank"] = (
        selection_summary[metric]
        .rank(method="average", ascending=True)
    )

rank_columns = [f"{metric}_rank" for metric in selection_metrics]
selection_summary["mean_rank"] = selection_summary[rank_columns].mean(axis=1)
selection_summary = selection_summary.sort_values("mean_rank")

selected_method = selection_summary.index[0]

display(selection_summary)
print("Выбранный метод:", selected_method)

## 9. Выбор степени балансировки

Для выбранного SMOTENC отдельно сравниваются отношения умерших к выжившим `0,10`, `0,20` и `0,33`. Выбирается минимальная степень балансировки, которая проходит упрощённые пороги статистической близости и технической корректности.

In [ ]:
ratio_candidates = [0.10, 0.20, 0.33]
ratio_rows = []

for sampling_strategy in ratio_candidates:
    for run in range(N_RUNS):
        ratio_rows.append(
            evaluate_generation(
                method=selected_method,
                seed=BASE_SEED + run,
                sampling_strategy=sampling_strategy,
            )
        )

ratio_results = pd.DataFrame(ratio_rows)

ratio_summary = (
    ratio_results
    .groupby("sampling_strategy", observed=True)[
        [
            *selection_metrics,
            "duplicate_rate",
            "range_violations",
            "invalid_binary",
            "synthetic_carotid_share",
        ]
    ]
    .median()
)

display(ratio_summary)

In [ ]:
ratio_limits = {
    "smd_mean": 0.25,
    "wasserstein_mean": 0.50,
    "categorical_difference": 0.30,
    "missingness_difference": 0.15,
    "correlation_difference": 0.35,
    "effect_difference": 0.35,
}

ratio_summary["passes_checks"] = True

for metric, upper_limit in ratio_limits.items():
    ratio_summary["passes_checks"] &= (
        ratio_summary[metric] <= upper_limit
    )

ratio_summary["passes_checks"] &= (
    (ratio_summary["range_violations"] == 0)
    & (ratio_summary["invalid_binary"] == 0)
)

passing_ratios = ratio_summary.index[
    ratio_summary["passes_checks"]
].tolist()

if passing_ratios:
    selected_sampling_strategy = float(min(passing_ratios))
else:
    fallback_summary = ratio_summary.copy()
    for metric in selection_metrics:
        fallback_summary[f"{metric}_rank"] = (
            fallback_summary[metric].rank(ascending=True)
        )
    fallback_summary["mean_rank"] = fallback_summary[
        [f"{metric}_rank" for metric in selection_metrics]
    ].mean(axis=1)
    selected_sampling_strategy = float(
        fallback_summary["mean_rank"].idxmin()
    )

selected_runs = ratio_results.loc[
    ratio_results["sampling_strategy"].eq(selected_sampling_strategy)
].copy()

method_medians = selected_runs[selection_metrics].median()
method_iqr = (
    selected_runs[selection_metrics].quantile(0.75)
    - selected_runs[selection_metrics].quantile(0.25)
).replace(0, 1)

selected_runs["distance_to_typical_run"] = (
    (selected_runs[selection_metrics] - method_medians)
    .abs()
    .div(method_iqr)
    .sum(axis=1)
)

selected_seed = int(
    selected_runs
    .sort_values("distance_to_typical_run")
    .iloc[0]["seed"]
)

display(ratio_summary)
print("Выбранное отношение умерших к выжившим:", selected_sampling_strategy)
print("Типичный random_state:", selected_seed)

ratio_plot_frame = ratio_results.melt(
    id_vars=["sampling_strategy"],
    value_vars=selection_metrics,
    var_name="metric",
    value_name="value",
)

figure, axes = plt.subplots(2, 3, figsize=(18, 10))

for axis, metric in zip(axes.flat, selection_metrics):
    sns.boxplot(
        data=ratio_plot_frame.loc[ratio_plot_frame["metric"] == metric],
        x="sampling_strategy",
        y="value",
        hue="sampling_strategy",
        palette="Blues",
        legend=False,
        ax=axis,
    )
    axis.set_title(metric)
    axis.set_xlabel("Отношение умерших к выжившим")

figure.suptitle("Сравнение степени балансировки", y=1.02)
figure.tight_layout()
plt.show()
plt.close(figure)

## 10. Итоговый сбалансированный датасет

In [ ]:
balanced_final, synthetic_final = generate_balanced_dataset(
    selected_method,
    selected_seed,
    selected_sampling_strategy,
)

final_overview = pd.DataFrame(
    {
        "Показатель": [
            "Всего строк",
            "Реальных пациентов",
            "Синтетических пациентов",
            "Выживших",
            "Умерших всего",
            "Реальных умерших",
            "Синтетических умерших",
            "Отношение умерших к выжившим",
        ],
        "Значение": [
            len(balanced_final),
            int((balanced_final["is_synthetic"] == 0).sum()),
            int((balanced_final["is_synthetic"] == 1).sum()),
            int((balanced_final["hospital_death"] == 0).sum()),
            int((balanced_final["hospital_death"] == 1).sum()),
            int(
                (
                    (balanced_final["hospital_death"] == 1)
                    & (balanced_final["is_synthetic"] == 0)
                ).sum()
            ),
            int(
                (
                    (balanced_final["hospital_death"] == 1)
                    & (balanced_final["is_synthetic"] == 1)
                ).sum()
            ),
            (
                (balanced_final["hospital_death"] == 1).sum()
                / (balanced_final["hospital_death"] == 0).sum()
            ),
        ],
    }
)

display(final_overview)
display(
    balanced_final
    .groupby(["vascular_group", "is_synthetic"], observed=True)
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "real", 1: "synthetic"})
)

In [ ]:
balanced_final.to_csv(
    FINAL_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Сохранён итоговый файл:", FINAL_PATH.resolve())

## 11. Статистические показатели до и после балансировки

Таблицы показывают изменение величин эффектов и p-value. p-value сбалансированного датасета являются исследовательскими: синтетические строки не считаются новой независимой клинической информацией.

In [ ]:
continuous_comparison_rows = []

for variable in continuous_analysis_features:
    real_alive = real_data.loc[
        real_data["hospital_death"] == 0,
        variable,
    ].dropna()
    real_deceased = real_data.loc[
        real_data["hospital_death"] == 1,
        variable,
    ].dropna()
    balanced_alive = balanced_final.loc[
        balanced_final["hospital_death"] == 0,
        variable,
    ].dropna()
    balanced_deceased = balanced_final.loc[
        balanced_final["hospital_death"] == 1,
        variable,
    ].dropna()

    real_test = mannwhitneyu(
        real_deceased,
        real_alive,
        alternative="two-sided",
    )
    balanced_test = mannwhitneyu(
        balanced_deceased,
        balanced_alive,
        alternative="two-sided",
    )

    continuous_comparison_rows.append(
        {
            "variable": variable,
            "label_ru": feature_labels[variable],
            "real_deceased_median": real_deceased.median(),
            "synthetic_deceased_median": synthetic_final[variable].median(),
            "effect_real": rank_biserial_effect(real_data, variable),
            "effect_balanced": rank_biserial_effect(balanced_final, variable),
            "p_real": real_test.pvalue,
            "p_balanced_exploratory": balanced_test.pvalue,
        }
    )

continuous_comparison = pd.DataFrame(continuous_comparison_rows)
display(continuous_comparison)

In [ ]:
binary_comparison_rows = []

for variable in binary_analysis_features:
    real_table = pd.crosstab(
        real_data[variable],
        real_data["hospital_death"],
    ).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    balanced_table = pd.crosstab(
        balanced_final[variable],
        balanced_final["hospital_death"],
    ).reindex(index=[0, 1], columns=[0, 1], fill_value=0)

    real_test = fisher_exact(real_table.to_numpy())
    balanced_test = fisher_exact(balanced_table.to_numpy())

    binary_comparison_rows.append(
        {
            "variable": variable,
            "label_ru": feature_labels[variable],
            "real_deceased_prevalence": real_data.loc[
                real_data["hospital_death"] == 1,
                variable,
            ].mean(),
            "synthetic_deceased_prevalence": synthetic_final[variable].mean(),
            "log_or_real": log_odds_effect(real_data, variable),
            "log_or_balanced": log_odds_effect(balanced_final, variable),
            "p_real": real_test.pvalue,
            "p_balanced_exploratory": balanced_test.pvalue,
        }
    )

binary_comparison = pd.DataFrame(binary_comparison_rows)
display(binary_comparison)

## 12. Графическое сравнение реальных и синтетических умерших

In [ ]:
plot_frame = pd.concat(
    [
        real_deaths.assign(source="Реальные умершие"),
        synthetic_final.assign(source="Синтетические умершие"),
    ],
    ignore_index=True,
)

figure, axes = plt.subplots(3, 3, figsize=(18, 14))

for axis, variable in zip(axes.flat, continuous_analysis_features):
    sns.boxplot(
        data=plot_frame,
        x="source",
        y=variable,
        hue="source",
        palette={
            "Реальные умершие": "#C06C84",
            "Синтетические умершие": "#6C9A8B",
        },
        legend=False,
        ax=axis,
    )
    axis.set_title(feature_labels[variable])
    axis.set_xlabel("")
    axis.set_ylabel("")
    axis.tick_params(axis="x", rotation=10)

figure.suptitle(
    f"Реальные и синтетические умершие: {selected_method}",
    y=1.02,
)
figure.tight_layout()
plt.show()
plt.close(figure)

In [ ]:
categorical_plot_rows = []

for variable in categorical_analysis_features:
    for source_label, frame in [
        ("Реальные умершие", real_deaths),
        ("Синтетические умершие", synthetic_final),
    ]:
        distribution = (
            frame[variable]
            .astype("string")
            .fillna("<NA>")
            .value_counts(normalize=True)
        )
        for category, proportion in distribution.items():
            categorical_plot_rows.append(
                {
                    "variable": variable,
                    "label_ru": feature_labels[variable],
                    "category": category,
                    "source": source_label,
                    "proportion": proportion,
                }
            )

categorical_comparison = pd.DataFrame(categorical_plot_rows)
display(categorical_comparison)

## 13. Итоговый вывод

In [ ]:
selected_summary = (
    ratio_results
    .loc[
        ratio_results["sampling_strategy"].eq(selected_sampling_strategy),
        summary_metrics,
    ]
    .median()
)

display(
    pd.DataFrame(
        {
            "Критерий": [
                "Выбранный метод",
                "Random state итоговой реализации",
                "Выбранное отношение умерших к выжившим",
                "Медианный абсолютный SMD",
                "Медианное нормированное расстояние Вассерштейна",
                "Медианное различие категориальных распределений",
                "Медианное различие корреляций",
                "Медианное различие эффектов",
                "Медианная доля полных копий",
                "Медианное число нарушений диапазонов",
            ],
            "Значение": [
                selected_method,
                selected_seed,
                selected_sampling_strategy,
                selected_summary["smd_mean"],
                selected_summary["wasserstein_mean"],
                selected_summary["categorical_difference"],
                selected_summary["correlation_difference"],
                selected_summary["effect_difference"],
                selected_summary["duplicate_rate"],
                selected_summary["range_violations"],
            ],
        }
    )
)

display(
    Markdown(
        f"""
Выбран **{selected_method}** с типичным `random_state={selected_seed}`.  
В итоговый набор добавлено **{len(synthetic_final)}** синтетических умерших; всего стало **{int(balanced_final['hospital_death'].sum())}** умерших и **{int((balanced_final['hospital_death'] == 0).sum())}** выживших.  
Итоговое отношение умерших к выжившим: **{balanced_final['hospital_death'].sum() / (balanced_final['hospital_death'] == 0).sum():.3f}**.  
Сбалансированные p-value следует использовать только как исследовательские; основные клинические выводы сохраняются по реальным данным.
"""
    )
)